# AMPS II · Week 01 — Lab
## Linear elasticity, locking, algebraic multigrid, reaction forces, hyperelasticity

| | |
|---|---|
| **Lecture sections** | Part 0 (module pattern) · Part A–B (elasticity, locking, GAMG, reactions) · Part C (neo-Hookean, load steps) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | write a model module and import it from MPI cells · see P1 lock in bending · measure what the rigid-body near-nullspace does for GAMG · compute reaction forces two ways · run a neo-Hookean cantilever with load steps |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. In this course the model lives in a
module written once with `%%writefile` (§1) and imported by every `%%fenicsx` cell. Results pass through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture Part 0, §B1)*
Read it once: parameters, mesh with facet tags, $\boldsymbol\sigma(\mathbf u)$, the rigid-body modes, a CG + GAMG solve that returns the tip deflection,
and the residual on the clamped DOFs. Run the cell — it only writes the file.

In [ ]:
%%writefile /tmp/fnx/beam.py
"""Cantilever L x B x H, clamped at x = 0, total shear load P on x = L. Linear elasticity, CG + GAMG, reactions."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh, la
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

E, nu = 1.0, 0.3
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
eps = lambda u: ufl.sym(ufl.grad(u))
sigma = lambda u: 2 * mu * eps(u) + lmbda * ufl.tr(eps(u)) * ufl.Identity(3)

def timoshenko(P, L, B=1.0, H=1.0):
    """tip deflection of a Timoshenko cantilever (Cowper's shear coefficient)"""
    I = B * H**3 / 12; kap = 10 * (1 + nu) / (12 + 11 * nu)
    return P * L**3 / (3 * E * I) + P * L / (kap * mu * B * H)

def make_beam(nx, L=10.0, B=1.0, H=1.0, ny=None):
    """tetrahedral mesh nx x ny x ny (default ny = nx * H / L, at least 1); facet tags: 1 clamp (x = 0), 2 loaded end (x = L)"""
    ny = ny or max(1, round(nx * H / L))
    msh = mesh.create_box(comm, [[0, 0, 0], [L, B, H]], [nx, ny, ny], cell_type=mesh.CellType.tetrahedron)
    left = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], 0.0))
    right = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[0], L))
    fac = np.r_[left, right]; o = np.argsort(fac)
    ft = mesh.meshtags(msh, 2, fac[o], np.r_[np.full(len(left), 1), np.full(len(right), 2)][o].astype(np.int32))
    return msh, left, ufl.Measure("ds", domain=msh, subdomain_data=ft)

def rigid_body_modes(V):
    """the six rigid-body motions as a PETSc near-nullspace (API sheet §16b)"""
    x = V.tabulate_dof_coordinates()
    vecs = [la.vector(V.dofmap.index_map, V.dofmap.index_map_bs) for _ in range(6)]
    b = [w.array.reshape(-1, 3) for w in vecs]
    for i in range(3):
        b[i][:, i] = 1.0
    b[3][:, 0], b[3][:, 1] = -x[:, 1], x[:, 0]
    b[4][:, 0], b[4][:, 2] = x[:, 2], -x[:, 0]
    b[5][:, 2], b[5][:, 1] = x[:, 1], -x[:, 2]
    la.orthonormalize(vecs)
    return PETSc.NullSpace().create(vectors=[w.petsc_vec for w in vecs])

def solve_cantilever(nx, deg, P=1e-3, L=10.0, B=1.0, H=1.0, ny=None, nullspace=True, pc="gamg", rtol=1e-10):
    """returns (uh, info); info: dofs, CG iterations, converged reason, tip deflection (average u_z on x = L, positive down)"""
    msh, left, ds = make_beam(nx, L, B, H, ny)
    V = fem.functionspace(msh, ("Lagrange", deg, (3,)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    t = fem.Constant(msh, np.array([0.0, 0.0, -P / (B * H)]))
    a = fem.form(ufl.inner(sigma(u), eps(v)) * ufl.dx); Lf = fem.form(ufl.dot(t, v) * ds(2))
    bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, left), V)
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    if nullspace:
        A.setNearNullSpace(rigid_body_modes(V))
    b = assemble_vector(Lf); apply_lifting(b, [a], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, [bc])
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("cg"); ksp.getPC().setType(pc)
    ksp.setTolerances(rtol=rtol, max_it=5000)
    uh = fem.Function(V); ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
    tip = -comm.allreduce(fem.assemble_scalar(fem.form(uh[2] * ds(2))), op=MPI.SUM) / (B * H)
    info = dict(dofs=V.dofmap.index_map.size_global * 3, its=ksp.getIterationNumber(), reason=ksp.getConvergedReason(), tip=tip)
    return uh, dict(info, a=a, L=Lf, left=left, ds=ds, msh=msh)

def residual_on_clamp(uh, info):
    """r = K u - f with K and f assembled WITHOUT boundary conditions; returns (coordinates, r) of the clamped DOFs owned here"""
    V = uh.function_space
    A0 = assemble_matrix(info["a"]); A0.assemble()
    f0 = assemble_vector(info["L"]); f0.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    r = A0.createVecLeft(); A0.mult(uh.x.petsc_vec, r); r.axpy(-1.0, f0)
    dofs = fem.locate_dofs_topological(V, 2, info["left"]); dofs = dofs[dofs < V.dofmap.index_map.size_local]
    return V.tabulate_dof_coordinates()[dofs], r.array.reshape(-1, 3)[dofs]

def neo_hookean_cantilever(P, nsteps, nx=20, deg=2, L=10.0, B=1.0, H=1.0, linesearch="none"):
    """dead tip load P in nsteps equal steps; returns [(load, Newton its, reason, tip deflection)]"""
    from dolfinx.fem.petsc import NonlinearProblem
    msh, left, ds = make_beam(nx, L, B, H)
    V = fem.functionspace(msh, ("Lagrange", deg, (3,))); u = fem.Function(V); v = ufl.TestFunction(V)
    bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, left), V)
    F = ufl.Identity(3) + ufl.grad(u); C = F.T * F; J = ufl.det(F)
    psi = mu / 2 * (ufl.tr(C) - 3) - mu * ufl.ln(J) + lmbda / 2 * ufl.ln(J) ** 2
    t = fem.Constant(msh, np.zeros(3))
    R = ufl.derivative(psi * ufl.dx - ufl.dot(t, u) * ds(2), u, v)
    pr = NonlinearProblem(R, u, bcs=[bc], petsc_options_prefix=f"nh{nsteps}_{linesearch}_",
                          petsc_options={"snes_type": "newtonls", "snes_linesearch_type": linesearch, "snes_rtol": 1e-10,
                                         "snes_atol": 1e-12, "snes_max_it": 30, "ksp_type": "preonly", "pc_type": "lu",
                                         "pc_factor_mat_solver_type": "mumps"})
    out = []
    for k in range(1, nsteps + 1):
        t.value = np.array([0.0, 0.0, -k / nsteps * P / (B * H)])
        pr.solve(); s = pr.solver
        tip = -comm.allreduce(fem.assemble_scalar(fem.form(u[2] * ds(2))), op=MPI.SUM) / (B * H)
        out.append((k / nsteps * P, s.getIterationNumber(), s.getConvergedReason(), tip))
        if s.getConvergedReason() <= 0:
            break
    return out

## 2 · Locking: P1 versus P2  *(lecture §A6, §B2)*
Cantilever $10\times1\times1$, $E=1$, $\nu=0.3$, tip load $P=10^{-3}$. Compare the tip deflection with Timoshenko beam theory (`timoshenko(P, L)`).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from beam import *
rows = []
for deg, nxs in ((1, (10, 20, 40, 80)), (2, (10, 20, 40))):
    for nx in nxs:
        _, info = solve_cantilever(nx, deg)
        rows.append([deg, info["dofs"], info["tip"] / timoshenko(1e-3, 10.0), info["its"]])
        say(f"P{deg} nx={nx:3d}  DOFs {info['dofs']:6d}  tip/Timoshenko {rows[-1][2]:.4f}  CG its {info['its']}")
if comm.rank == 0:
    json.dump(rows, open("/tmp/fnx/w01_locking.json", "w"))

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
R = np.array(json.load(open("/tmp/fnx/w01_locking.json")))
fig, ax = plt.subplots(figsize=(6, 4))
for deg, mk in ((1, "o-"), (2, "s-")):
    m = R[:, 0] == deg
    ax.semilogx(R[m, 1], R[m, 2], mk, lw=2, label=f"P{deg}")
ax.axhline(1, color="k", ls="--", lw=1, label="Timoshenko")
ax.set(xlabel="DOFs", ylabel="tip / Timoshenko", ylim=(0, 1.1)); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** At (about) the same number of DOFs, how far apart are P1 and P2? Why does P1 converge at all, and why so slowly?
> Why does the converged P2 value stay slightly **below** 1?

> *your answer*

## 3 · CG + algebraic multigrid: the near-nullspace  *(lecture §B3–B4)*

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from beam import *
for nx in (20, 40, 80):
    _, a = solve_cantilever(nx, 1, nullspace=True)
    _, b = solve_cantilever(nx, 1, nullspace=False)
    say(f"P1 nx={nx:3d} DOFs {a['dofs']:6d}:  with rigid-body modes {a['its']:3d} CG its,  without {b['its']:3d}  (reasons {a['reason']}, {b['reason']})")

In [ ]:
%%fenicsx -np 2
# ✏️ Your turn (5 min): replace GAMG by a Jacobi preconditioner (pc="jacobi") for nx = 20, 40, 80 and compare the iteration counts.
# TODO

## 4 · Reaction forces two ways  *(lecture §B5–B6)*
The clamp must carry the load: $R_z=P$. Compare the consistent reaction (residual of the unconstrained system) with the flux integral
$\int_{\Gamma_D}(\boldsymbol\sigma(\mathbf u_h)\mathbf n)_z\,ds$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from beam import *
P = 1e-3
for deg, nx in ((1, 20), (1, 40), (2, 10), (2, 20)):
    uh, info = solve_cantilever(nx, deg, P=P)
    X, r = residual_on_clamp(uh, info)
    Rz = comm.allreduce(r[:, 2].sum(), op=MPI.SUM)
    n = ufl.FacetNormal(info["msh"])
    Rf = comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(sigma(uh), n)[2] * info["ds"](1))), op=MPI.SUM)
    say(f"P{deg} nx={nx:3d}:  consistent R_z/P = {Rz / P:.10f}   flux R_z/P = {Rf / P:.4f}")

> **✏️ Your turn (5 min).** Which components of the consistent reaction force should vanish? Check $R_x$ and $R_y$ by editing the cell above.
> Why is the consistent value exact up to the solver tolerance, and the flux not?

> *your answer*

## 5 · Neo-Hookean cantilever: load steps  *(lecture §C1–C5)*

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from beam import *
small = neo_hookean_cantilever(1e-6, 1, nx=10)
_, lin = solve_cantilever(10, 2, P=1e-6)
say(f"small-load check (P = 1e-6): neo-Hookean tip {small[-1][3]:.6e}, linear P2 tip {lin['tip']:.6e}")
for n in (1, 4):
    out = neo_hookean_cantilever(4e-3, n, nx=10)
    say(f"P = 4e-3 in {n} step(s): Newton its {[o[1] for o in out]}, reasons {[o[2] for o in out]}, tip {out[-1][3]:.4f} "
        f"(linear theory {timoshenko(out[-1][0], 10.0):.2f})")

> **✏️ Your turn (5 min).** Why does the one-step solve stop with reason −4? What does the small-load check verify, and what does it **not** verify?

> *your answer*

## 6 · LLM exercise: a module and its tests  *(lecture Part D)*
Ask your LLM for a module `plate.py` (a 3D plate $4\times4\times0.2$ clamped on one edge, uniform pressure on the top face) with functions that return
numbers, **and** for a test cell. Paste the API sheet (§16) and state DOLFINx 0.11. A prompt to start from:

```text
Write a DOLFINx 0.11 module (as an %%writefile /tmp/fnx/plate.py cell) for a clamped 3D elastic plate 4 x 4 x 0.2 under uniform pressure p on
the top face, P2 tetrahedra, CG + GAMG with the rigid-body near-nullspace. Functions: make_mesh(n), solve(n, p) -> (uh, info) with the maximum
deflection, CG iterations and the converged reason. Then a %%fenicsx -np 2 test cell that checks: the consistent reaction equals p * area,
the deflection is linear in p, and -np 1 and -np 2 give the same numbers. Use the attached API sheet.
```

```python
# paste here
```

**Check before you trust it**

- [ ] DOLFINx 0.11 API (no `VectorFunctionSpace`, no `uh.vector`, `petsc_options_prefix` given)
- [ ] The near-nullspace is attached **before** the solve, and the iteration count is printed
- [ ] The reaction comes from the residual of the unconstrained system
- [ ] Every global number uses `allreduce`; printing only on rank 0
- [ ] The test cell actually fails when you break the model on purpose (e.g. double the pressure in one place)